# 03 — Sentiment Analysis

## Objective
Classify each comment as positive, negative, or neutral using a state-of-the-art BERT-based model, and compare against a simpler baseline.

## Model selection

| Language | Primary model | Baseline |
|----------|--------------|----------|
| Spanish | `pysentimiento` (BERT fine-tuned on Spanish social media) | Polarity lexicon (NRC-EmoLex ES) |
| English | `cardiffnlp/twitter-roberta-base-sentiment` | VADER |

**Why these choices?**
- pysentimiento is the best publicly available Spanish social-media sentiment model.
- RoBERTa-base is the de facto standard for English Twitter sentiment.
- VADER is English-only; TextBlob-es is unmaintained, so a transparent polarity lexicon is used for the Spanish baseline.

## Business value
Sentiment classification converts 10,000+ raw comments into a single KPI: *What % of fans feel positively about our team today?* This is the core metric the dashboard tracks over time.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.sentiment import predict_sentiment, add_sentiment_to_dataframe
from src.utils import load_dataframe, save_dataframe

import pandas as pd
import matplotlib.pyplot as plt

### Load preprocessed data

In [ ]:
df = load_dataframe(str(project_root / 'data' / 'processed' / 'preprocessed.parquet'))
print(f"Loaded {len(df)} preprocessed comments")
display(df[['text_clean', 'language', 'teams']].head(3))

### Run primary (transformer) model

In [ ]:
cached_path = project_root / 'data' / 'processed' / 'sentiment.parquet'
if cached_path.exists():
    df = load_dataframe(str(cached_path))
    print("Loaded cached sentiment data")
else:
    df = add_sentiment_to_dataframe(df, model='transformer')
    save_dataframe(df, str(project_root / 'data' / 'processed' / 'sentiment'), format='parquet')

display(df[['text_clean', 'sentiment_label', 'sentiment_positive', 'sentiment_negative']].head(10))

### Sentiment distribution

In [ ]:
if 'sentiment_label' in df.columns:
    dist = df['sentiment_label'].value_counts()
    display(dist)
    dist.plot(kind='bar', title='Sentiment Distribution', color=['green', 'red', 'gray'])
    plt.xticks(rotation=0)
    plt.show()
    
    if 'teams' in df.columns:
        team_sent = df.groupby('teams')['sentiment_label'].value_counts(normalize=True).unstack()
        team_sent.plot(kind='bar', stacked=True, title='Sentiment by Team')
        plt.xticks(rotation=45)
        plt.legend(title='Sentiment')
        plt.show()

### Baseline comparison

In [ ]:
if 'sentiment_baseline_label' in df.columns:
    agreement = (df['sentiment_label'] == df['sentiment_baseline_label']).mean()
    print(f"Transformer ↔ Baseline agreement: {agreement:.1%}")
    
    cross = pd.crosstab(
        df['sentiment_label'],
        df['sentiment_baseline_label'],
        normalize='index',
    )
    display(cross)

---
**Next step**: [04 — Topic Modeling & NER](04_topic_modeling_ner.ipynb)